# Projeto Completo - Classificador de Imagens
## Imports e funções

In [ ]:
import time 
import joblib

import pandas   as pd
import numpy    as np

from numpy                      import mean
from numpy                      import std
from sklearn                    import metrics
from sklearn.metrics            import confusion_matrix, f1_score
from sklearn.preprocessing      import minmax_scale

from sklearn.model_selection    import train_test_split, KFold, cross_val_score, cross_val_predict, ParameterGrid

from sklearn.naive_bayes        import GaussianNB, MultinomialNB, ComplementNB
from sklearn.neural_network     import MLPClassifier
from sklearn.neighbors          import KNeighborsClassifier
from sklearn.tree               import DecisionTreeClassifier

import traceback
import warnings
from sklearn.exceptions         import ConvergenceWarning
from datetime                   import datetime
# Parametros gerais para homogeneidade de metodologia
# Tamanho do conjunto de teste
tamanho_ds_teste=0.2
semente_aleatoria=42
versao_execucao = f"v{datetime.now().strftime('%Y%m%d-%H%M')}"

def raca_para_especie(raca):
    if raca in ['basset_hound', 'saint_bernard']:
        return 'dog'
    elif raca in ['Birman', 'Persian']:
        return 'cat'
    else:
        return raca  # fallback

def separar_dataset(df, scale=False):
    X = df.iloc[:, :-1]
    ### Para Bases com PCA que geram valores negativos permite normalizar os valores
    if scale:
        X = minmax_scale(X)
    y = df.iloc[:, -1]
    return X, y

## Parametros de configuração de GridSearch para os modelos

In [2]:
        # 'param_grid': {
        #     'hidden_layer_sizes': [ (50), (100), (150), (100, 50), (150, 75), (200, 100), (200, 100, 50), (150, 75, 25) ],
        #     'activation': ['identity', 'logistic', 'tanh', 'relu'],
        #     'solver': ['adam', 'sgd'],
        #     'learning_rate_init': [0.001, 0.01, 0.05],
        #     'max_iter': [1000, 1500, 2000, 2500, 3000], 'early_stopping': True
        # },
# param_grid = {
#     'hidden_layer_sizes': [(50), (100), (120), (150), (70,70), (100,50)],
#     'activation': ['identity', 'logistic', 'tanh', 'relu'],
#     'solver': ['adam', 'sgd'],
#     'learning_rate_init': [0.0001, 0.001, 0.01, 0.1],
#     'max_iter': [500, 1000, 1500, 2000]
# }
# Estrutura de configuração padronizada para todos os modelos
MODEL_CONFIGS = {

    'knn': {
        'class': KNeighborsClassifier,
        'param_grid': {
            'n_neighbors': [1, 3, 5, 7, 9, 11, 13, 15],
            'metric': ['euclidean', 'manhattan', 'chebyshev'],
            'weights': ['uniform', 'distance']
        },
        'fixed_params': {},
        'scale_data': False,
        'warning_exceptions': [],
        'model_name': 'KNN'
    },

    'dtree': {
        'class': DecisionTreeClassifier,
        'param_grid': {
            'criterion': ['gini', 'entropy', 'log_loss'],
            'max_depth': [2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]
        },
        'fixed_params': {},
        'scale_data': False,
        'warning_exceptions': [],
        'model_name': 'Decision Tree'
    },
    
    'nb': {
        'class': [GaussianNB, MultinomialNB, ComplementNB],
        'param_grid': {},  # NB não usa grid search normal - testa apenas os 3 tipos
        'fixed_params': {},
        'scale_data': 'nb_conditional',  # GaussianNB=False, outros=PCA only
        'warning_exceptions': [],
        'model_name': 'Naive Bayes'
    },
    
    'mlp': {
        'class': MLPClassifier,
        'param_grid': {
            'hidden_layer_sizes': [ (50), (100), (150), (100, 50), (150, 75), (200, 100), (200, 75, 25), (200, 100, 50) ],
            'activation': ['identity', 'logistic', 'tanh', 'relu'],
            'solver': ['adam', 'sgd'],
            'learning_rate_init': [0.001, 0.005, 0.01],
            'max_iter': [500, 1000, 1500],
        },
        'fixed_params': {
            'random_state': semente_aleatoria,
        },
        'scale_data': False,
        'warning_exceptions': [ConvergenceWarning],
        'model_name': 'MLP'
    },       

    # 'bagging': {
    #     'class': BaggingClassifier,
    #     'param_grid': {
    #         'n_estimators': [10, 50, 100],
    #         'max_samples': [0.5, 0.7, 1.0],
    #         'max_features': [0.5, 0.7, 1.0]
    #     },
    #     'fixed_params': {'random_state': semente_aleatoria},
    #     'scale_data': False,
    #     'warning_exceptions': [],
    #     'model_name': 'Bagging'
    # },
    
    # 'rf': {
    #     'class': RandomForestClassifier,
    #     'param_grid': {
    #         'n_estimators': [50, 100, 200],
    #         'max_depth': [5, 10, 15, None],
    #         'min_samples_split': [2, 5, 10],
    #         'min_samples_leaf': [1, 2, 4]
    #     },
    #     'fixed_params': {'random_state': semente_aleatoria},
    #     'scale_data': False,
    #     'warning_exceptions': [],
    #     'model_name': 'Random Forest'
    # },
    
    # 'stacking': {
    #     'class': StackingClassifier,
    #     'param_grid': {
    #         'estimators': [
    #             [('dt', DecisionTreeClassifier(random_state=semente_aleatoria)),
    #              ('nb', GaussianNB()),
    #              ('knn', KNeighborsClassifier())]
    #         ],
    #         'final_estimator': [LogisticRegression(random_state=semente_aleatoria), 
    #                            DecisionTreeClassifier(random_state=semente_aleatoria)]
    #     },
    #     'fixed_params': {},
    #     'scale_data': False,
    #     'warning_exceptions': [],
    #     'model_name': 'Stacking'
    # },
    
    # 'voting': {
    #     'class': VotingClassifier,
    #     'param_grid': {
    #         'voting': ['hard', 'soft']
    #     },
    #     'fixed_params': {
    #         'estimators': [
    #             ('dt', DecisionTreeClassifier(random_state=semente_aleatoria)),
    #             ('nb', GaussianNB()),
    #             ('knn', KNeighborsClassifier())
    #         ]
    #     },
    #     'scale_data': False,
    #     'warning_exceptions': [],
    #     'model_name': 'Voting'
    # }
}

In [3]:

TRAINING_TYPES = ["holdout", "crossvalidation"]

## Lendo lista de arquivos a processar

In [4]:
datafiles = pd.read_csv('../dataset_list.csv',encoding='utf-8')

datafiles.head(20)

,key,filename
0,hogfeat_128_16_4_9_pca,../Aula11/hogfeat_128_16_4_9_pca.csv.gz
1,hogfeat_256_64_2_18,../Aula11/hogfeat_256_64_2_18.csv.gz
2,hogfeat_256_64_2_9,../Aula11/hogfeat_256_64_2_9.csv.gz
3,hogfeat_128_16_4_9,../Aula11/hogfeat_128_16_4_9.csv.gz
4,hogfeat_256_32_2_9_pca,../Aula11/hogfeat_256_32_2_9_pca.csv.gz
5,hogfeat_128_16_2_9_pca,../Aula11/hogfeat_128_16_2_9_pca.csv.gz
6,hogfeat_128_32_2_9,../Aula11/hogfeat_128_32_2_9.csv.gz
7,lbpfeat_256_6_48,../Aula11/lbpfeat_256_6_48.csv.gz
8,lbpfeat_256_12_96,../Aula11/lbpfeat_256_12_96.csv.gz
9,hogfeat_256_32_2_9,../Aula11/hogfeat_256_32_2_9.csv.gz


## Lendo Dataframes e ajustando dados

In [5]:

dfs = {}
shapes = []
last_col = []

for metadata in datafiles.itertuples():
    # Imprime o arquivo que está sendo lido
    #print(f"Lendo arquivo: {metadata.key}")
    # Carrega o DataFrame
    df = pd.read_csv(metadata.filename)
    
    # Aplica a função raca_para_especie na coluna raca
    if 'raca' in df.columns:
        df['especie'] = df['raca'].apply(raca_para_especie)
        df = df.drop('raca', axis=1)
    
    # Elimina a coluna nome_arquivo se existir
    if 'nome_arquivo' in df.columns:
        df = df.drop('nome_arquivo', axis=1)
    
    dfs[metadata.key] = df
    shapes.append(df.shape)
    # Pega a última coluna
    last_col.append(df.columns[-1:].tolist())

# Adiciona as colunas shape e last_col ao datafiles
datafiles['shape'] = shapes
datafiles['last_column'] = last_col

datafiles.head(12)    

,key,filename,shape,last_column
0,hogfeat_128_16_4_9_pca,../Aula11/hogfeat_128_16_4_9_pca.csv.gz,"(800, 104)",[especie]
1,hogfeat_256_64_2_18,../Aula11/hogfeat_256_64_2_18.csv.gz,"(800, 649)",[especie]
2,hogfeat_256_64_2_9,../Aula11/hogfeat_256_64_2_9.csv.gz,"(800, 325)",[especie]
3,hogfeat_128_16_4_9,../Aula11/hogfeat_128_16_4_9.csv.gz,"(800, 3601)",[especie]
4,hogfeat_256_32_2_9_pca,../Aula11/hogfeat_256_32_2_9_pca.csv.gz,"(800, 94)",[especie]
5,hogfeat_128_16_2_9_pca,../Aula11/hogfeat_128_16_2_9_pca.csv.gz,"(800, 118)",[especie]
6,hogfeat_128_32_2_9,../Aula11/hogfeat_128_32_2_9.csv.gz,"(800, 325)",[especie]
7,lbpfeat_256_6_48,../Aula11/lbpfeat_256_6_48.csv.gz,"(800, 51)",[especie]
8,lbpfeat_256_12_96,../Aula11/lbpfeat_256_12_96.csv.gz,"(800, 99)",[especie]
9,hogfeat_256_32_2_9,../Aula11/hogfeat_256_32_2_9.csv.gz,"(800, 1765)",[especie]


### Definindo função generica para GridSearch

In [6]:
def get_scale_setting(params, dataset_key):
    """
    Determina se deve escalar baseado no modelo e parâmetros
    """
    nb_type = params.get('nb_type')
    if nb_type in ['MultinomialNB', 'ComplementNB']:
        if dataset_key and dataset_key.endswith('_pca'):
            return True
    return False
            
def create_model_from_config(config, params):
    """
    Cria modelo baseado na configuração - unificado para todos os tipos
    """
    if isinstance(config['class'], list):
        # NB: seleciona classe baseada no nb_type
        nb_type = params.get('nb_type')
        class_map = {
            'GaussianNB': GaussianNB,
            'MultinomialNB': MultinomialNB,
            'ComplementNB': ComplementNB
        }
        model_class = class_map[nb_type]
        return model_class()
    else:
        # Modelos normais: usa parâmetros do grid
        model_params = {**config['fixed_params'], **params}
        return config['class'](**model_params)

def executar_grid_search(model_key, best_dataset_df, dataset_key):
    """
    Executa avaliação de hiperparâmetros para um modelo de machine learning.
    
    Realiza grid search ou avaliação de múltiplas variantes do modelo especificado,
    testando diferentes combinações de parâmetros na melhor base de dados disponível.
    Para modelos com param_grid vazio, testa múltiplas variantes (ex: tipos de NB).
    
    Parâmetros
    ----------
    model_key : str
        Chave do modelo no dicionário MODEL_CONFIGS (ex: 'knn', 'mlp', 'nb').
        Define qual modelo será avaliado e suas configurações.
        
    best_dataset_df : pandas.DataFrame
        DataFrame da melhor base de dados identificada para avaliação.
        Deve conter as features nas colunas e o target na última coluna.
    
    Retorna
    -------
    pandas.DataFrame
        DataFrame com TODAS as configurações testadas, ordenadas
        por F1-score decrescente. Inclui configurações que falharam.
        Colunas incluem:
        - 'params': dicionário com os parâmetros da configuração
        - 'f1_score': score F1-weighted obtido (NaN para falhas)
        - 'status': 'sucesso', 'warning', ou 'erro'
        - 'error_msg': mensagem de erro (se aplicável)
    
    Notas
    -----
    - Utiliza holdout com tamanho_ds_teste, atualmente 80% para treino e 20% para teste
    - Para MLP, configurações com warnings de convergência são rejeitadas
    - Para modelo NB testa MultinomialNB e ComplementNB aplicando scaling [-1,1] a [0,1]
    - Outros modelos usam grid search padrão via ParameterGrid
    """
    config = MODEL_CONFIGS[model_key]
    
    # Preparação unificada das combinações
    if not config['param_grid']:
        if isinstance(config['class'], list):
            param_combinations = [{'nb_type': cls.__name__} for cls in config['class']]
        else:
            param_combinations = [{}]
    else:
        param_combinations = list(ParameterGrid(config['param_grid']))
    
    print(f"Executando avaliação para {config['model_name']}...")
    print(f"Total de configurações a testar: {len(param_combinations)}")
    
    results = []
    erros_encontrados = 0
    warnings_encontrados = 0
    sucessos = 0
    
    ### MLP teve configurações selecionadas manualmente a partir de dois grid searchs:
    ### 1. Utilizando a melhor base de dados disponível
    ### 2. Utilizando a a base com maior número de atributos
    ### Foi utilizado isso por que as configurações de MLP são muito sensíveis aos dados de treino
    ### Bases com maior número de atributos apresentam um desempenho muito melhor com certas configurações
    ### enquanto bases com menor número de atributos apresentam um desempenho muito melhor com outras configurações

    ### Notamos que a variação do numeros de iterações interfere na seleção dos TOPN pois depois que a rede neural
    ### convergiu, a variação do número de iterações já não impacta o desempenho.

    ### Dessa forma, para cada conjunto de configurações, excluido o numero de iterações, foi escolhida aquela com
    ### menor numero de iterações.

    ### Foram selecionadas as configurações top1, top 4, top 6, top 8 e top 10 para cada base, perfazendo um total
    ### de 10 configurações.

    if model_key == 'mlp':
        sucessos = 10
        hardcoded_params = {
            'params': [
                {'activation': 'relu', 'hidden_layer_sizes': (200, 100), 'learning_rate_init': 0.005, 'max_iter': 500, 'solver': 'adam'},
                {'activation': 'logistic', 'hidden_layer_sizes': (200, 100, 50), 'learning_rate_init': 0.005, 'max_iter': 500, 'solver': 'adam'},
                {'activation': 'relu', 'hidden_layer_sizes': 50, 'learning_rate_init': 0.01, 'max_iter': 500, 'solver': 'adam'},
                {'activation': 'tanh', 'hidden_layer_sizes': (150, 75), 'learning_rate_init': 0.01, 'max_iter': 500, 'solver': 'adam'},
                {'activation': 'relu', 'hidden_layer_sizes': (200, 100), 'learning_rate_init': 0.01, 'max_iter': 500, 'solver': 'adam'},
                {'activation': 'relu', 'hidden_layer_sizes': (200, 100, 50), 'learning_rate_init': 0.01, 'max_iter': 500, 'solver': 'sgd'},
                {'activation': 'identity', 'hidden_layer_sizes': (200, 100, 50), 'learning_rate_init': 0.001, 'max_iter': 500, 'solver': 'adam'},
                {'activation': 'relu', 'hidden_layer_sizes': (200, 100, 50), 'learning_rate_init': 0.005, 'max_iter': 500, 'solver': 'adam'},
                {'activation': 'relu', 'hidden_layer_sizes': 150, 'learning_rate_init': 0.01, 'max_iter': 500, 'solver': 'adam'},
                {'activation': 'relu', 'hidden_layer_sizes': (200, 100), 'learning_rate_init': 0.001, 'max_iter': 1500, 'solver': 'adam'}
            ],
            'f1_score': [
                0.806835531221790, 
                0.787968430168477, 
                0.787162698412698, 
                0.781637931034482, 
                0.781396922286752, 
                0.775636392206159, 
                0.774642857142857, 
                0.771720647773279,
                0.769341060123471, 
                0.768905317845995 
            ]
        }
        for i, params in enumerate(hardcoded_params['params']):
            results.append({
                'params': params,
                'f1_score': hardcoded_params['f1_score'][i],
                'status': 'sucesso'
            })        
    else:
        for i, params in enumerate(param_combinations):
            print(f"Testando configuração {i+1}/{len(param_combinations)}: {params}")
            
            try:
                scale = get_scale_setting(params, dataset_key)
                X, y = separar_dataset(best_dataset_df, scale=scale)
                X_train, X_test, y_train, y_test = train_test_split(
                    X, y, test_size=tamanho_ds_teste, random_state=semente_aleatoria
                )
                
                model = create_model_from_config(config, params)
                
                with warnings.catch_warnings(record=True) as w:
                    warnings.simplefilter("always")
                    model.fit(X_train, y_train)
                    
                    has_warning = any(
                        any(issubclass(warning.category, exc) for warning in w)
                        for exc in config['warning_exceptions']
                    )
                
                y_pred = model.predict(X_test)
                f1 = f1_score(y_test, y_pred, average='weighted')
                
                should_accept = not (model_key == 'mlp' and has_warning)
                
                if should_accept:
                    results.append({
                        'params': params,
                        'f1_score': f1,
                        'status': 'warning' if has_warning else 'sucesso'
                    })
                    sucessos += 1
                
                marker = "⚠" if has_warning else "✓"
                print(f"\t{marker} Sucesso - F1: {f1:.4f}")
                
            except Exception as e:
                erro_detalhado = f"{type(e).__name__}: {str(e)}"
                print(f"\t✗ ERRO DETALHADO: {erro_detalhado}")
                print(f"\t   Parâmetros que causaram erro: {params}")
                traceback.print_exc()
                
                # Também mostrar os fixed_params se for modelo normal
                if not isinstance(config['class'], list):
                    print(f"\t   Fixed params: {config['fixed_params']}")
                
                erros_encontrados += 1
                continue
        
    # ordena resultados e prepara retorno
    results_df = pd.DataFrame(results)
    results_df = results_df.sort_values('f1_score', ascending=False)
    all_configs = results_df.copy()
    all_configs.reset_index(drop=True, inplace=True)

    # imprime relatório final
    print(f"\n{config['model_name']} - RELATÓRIO FINAL:")
    print("="*60)
    print(f"Total configurações: {len(param_combinations)}")
    print(f"✓ Sucessos: {sucessos}")
    print(f"⚠ Warnings: {warnings_encontrados}")
    print(f"✗ Erros...: {erros_encontrados}")
    print(f"Total configurações válidas: {len(all_configs)}")
      
    return all_configs

## Identificando as melhores 10 configurações

#### Utiliza o melhor modelo disponivel para identificar as top10 configurações.

In [7]:
def executar_all_grid_search(dataframe_zero, dataset_key):
    """
    Executa grid search para todos os modelos usando a primeira base de dados.
    
    Percorre todos os modelos definidos no MODEL_CONFIGS, executa grid search
    na base de dados fornecida e armazena os resultados.
    
    Retorna
    -------
    dict
        Dicionário com os resultados do grid search para cada modelo:
        {
            'model_key': {
                'all_configs': DataFrame com todos os resultados das configurações testadas,
                'model_name': str nome do modelo
            }
        }
    """
    
    # Usar primeira base como no código existente
    print(f"📊 Usando base de dados: {dataset_key}")
    print(f"📏 Shape: {dataframe_zero.shape}")
    print("=" * 60)
    
    # Dicionário para armazenar resultados
    resultados_grid_search = {}
    
    # Percorrer todos os modelos no MODEL_CONFIGS
    modelos_keys = list(MODEL_CONFIGS.keys())
    
    for i, model_key in enumerate(modelos_keys):
        config = MODEL_CONFIGS[model_key]
        print(f"\n🔧 MODELO {i+1}/{len(modelos_keys)}: {config['model_name']}")
        print("-" * 50)
        
        try:
            # Executar grid search para este modelo
            all_configs = executar_grid_search(model_key, dataframe_zero, dataset_key)
            
            # Armazenar resultado
            resultados_grid_search[model_key] = {
                'all_configs': all_configs,
                'model_name': config['model_name']
            }
            
            print(f"✅ Grid search concluído - {len(all_configs)} configurações analisadas")
            
        except Exception as e:
            traceback.print_exc()
            print(f"❌ ERRO no grid search de {config['model_name']}: {str(e)}")
            resultados_grid_search[model_key] = {
                'all_configs': pd.DataFrame(),
                'model_name': config['model_name'],
                'erro': str(e)
            }
       
    return resultados_grid_search


In [8]:

dataset_key = datafiles['key'][0]
dataframe_busca = dfs[dataset_key]

all_configs = executar_all_grid_search(dataframe_busca, dataset_key)


📊 Usando base de dados: hogfeat_128_16_4_9_pca
📏 Shape: (800, 104)

🔧 MODELO 1/4: KNN
--------------------------------------------------
Executando avaliação para KNN...
Total de configurações a testar: 48
Testando configuração 1/48: {'metric': 'euclidean', 'n_neighbors': 1, 'weights': 'uniform'}
	✓ Sucesso - F1: 0.7363
Testando configuração 2/48: {'metric': 'euclidean', 'n_neighbors': 1, 'weights': 'distance'}
	✓ Sucesso - F1: 0.7363
Testando configuração 3/48: {'metric': 'euclidean', 'n_neighbors': 3, 'weights': 'uniform'}
	✓ Sucesso - F1: 0.6799
Testando configuração 4/48: {'metric': 'euclidean', 'n_neighbors': 3, 'weights': 'distance'}
	✓ Sucesso - F1: 0.6799
Testando configuração 5/48: {'metric': 'euclidean', 'n_neighbors': 5, 'weights': 'uniform'}
	✓ Sucesso - F1: 0.7207
Testando configuração 6/48: {'metric': 'euclidean', 'n_neighbors': 5, 'weights': 'distance'}
	✓ Sucesso - F1: 0.7207
Testando configuração 7/48: {'metric': 'euclidean', 'n_neighbors': 7, 'weights': 'uniform'}
	✓ 

#### Filtrando as top10 configuracoes de cada modelo

In [9]:
# Código para filtrar apenas as top 10 configurações de cada model key
def filtrar_top_10_configs(all_configs):
    """
    Filtra apenas as top 10 configurações de cada model key baseado no f1_score.
    
    Args:
        all_configs (dict): Dicionário com as configurações de cada modelo
        
    Returns:
        dict: Dicionário filtrado com apenas as top 10 configurações por modelo
    """
    top_10_configs = {}
    
    for model_key, model_data in all_configs.items():
        # Copia os dados do modelo
        filtered_data = model_data.copy()
        
        # Filtra apenas as top 10 configurações baseado no f1_score
        if 'all_configs' in filtered_data and not filtered_data['all_configs'].empty:
            # Ordena por f1_score decrescente e mantém apenas as primeiras 10 linhas
            filtered_data['all_configs'] = (
                filtered_data['all_configs']
                .sort_values('f1_score', ascending=False)
                .head(10)
                .reset_index(drop=True)
            )
        
        top_10_configs[model_key] = filtered_data
        
    return top_10_configs

# Aplicar a filtragem
top10_configs = filtrar_top_10_configs(all_configs)


## TOP 10 configurações identificadas

In [10]:


from IPython.display import display

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', 0)

print("Top 10 Configurações:")
display(top10_configs)

Top 10 Configurações:


{'knn': {'all_configs':                                                               params  \
  0   {'metric': 'chebyshev', 'n_neighbors': 3, 'weights': 'distance'}   
  1    {'metric': 'chebyshev', 'n_neighbors': 3, 'weights': 'uniform'}   
  2   {'metric': 'chebyshev', 'n_neighbors': 5, 'weights': 'distance'}   
  3    {'metric': 'chebyshev', 'n_neighbors': 5, 'weights': 'uniform'}   
  4  {'metric': 'chebyshev', 'n_neighbors': 11, 'weights': 'distance'}   
  5   {'metric': 'chebyshev', 'n_neighbors': 11, 'weights': 'uniform'}   
  6  {'metric': 'chebyshev', 'n_neighbors': 15, 'weights': 'distance'}   
  7   {'metric': 'chebyshev', 'n_neighbors': 15, 'weights': 'uniform'}   
  8   {'metric': 'chebyshev', 'n_neighbors': 7, 'weights': 'distance'}   
  9    {'metric': 'chebyshev', 'n_neighbors': 7, 'weights': 'uniform'}   
  
     f1_score   status  
  0  0.767689  sucesso  
  1  0.767689  sucesso  
  2  0.762123  sucesso  
  3  0.762123  sucesso  
  4  0.759038  sucesso  
  5  0.7590

## Aplicando as top10 configurações nas 12 bases de dados

In [11]:
# Aplicando top 10 configurações de todos os modelos em todas as bases de dados
par_training = TRAINING_TYPES

resultados_todos_modelos = []

print("Aplicando top 10 configurações de todos os modelos em todas as bases de dados...")
print("=" * 80)

i=0
for metadata in datafiles.itertuples():
    i += 1
    print(f"\n📊 Dataset {i}: {metadata.key}")
    print("-" * 60)
    
    for model_key in top10_configs.keys():
        print(f"\n🔧 Modelo: {top10_configs[model_key]['model_name']} ({model_key}) Dataset {i}: {metadata.key}")
        
        model_top_configs = top10_configs[model_key]['all_configs']
        
        for config_idx, config_row in model_top_configs.iterrows():
            config_rank = config_idx + 1
            params = config_row['params']
            
            scale_data = get_scale_setting(params, metadata.key)
            print(f"\t\tEscalamento: {'Sim' if scale_data else 'Não'}")
            
            df = dfs[metadata.key]
            X, y = separar_dataset(df, scale=scale_data)
            
            model_config = MODEL_CONFIGS[model_key]
            model_class = model_config['class']
            
            for training in par_training:
                print(f"\t\tTraining: {training}")
                
                try:
                    if model_key == 'nb':
                        nb_type = params['nb_type']
                        if nb_type == 'GaussianNB':
                            model = GaussianNB()
                        elif nb_type == 'MultinomialNB':
                            model = MultinomialNB()
                        elif nb_type == 'ComplementNB':
                            model = ComplementNB()
                        else:
                            raise ValueError(f"Tipo NB desconhecido: {nb_type}")
                    else:
                        model_params = {**params, **model_config['fixed_params']}
                        model = model_class(**model_params)
                    
                    f1 = 0.0
                    f1_std = 0.0
                    confusao = np.array([])
                    execution_time = 0.0
                    
                    start_time = time.time()
                    
                    if training == "holdout":
                        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=tamanho_ds_teste, random_state=semente_aleatoria)
                        
                        with warnings.catch_warnings(record=True) as w:
                            warnings.simplefilter("always")
                            model.fit(X_train, y_train)
                            
                            warning_exceptions = model_config.get('warning_exceptions', [])
                            has_warning = any(issubclass(warning.category, tuple(warning_exceptions)) for warning in w)
                        
                        y_pred = model.predict(X_test)
                        f1 = f1_score(y_test, y_pred, average='weighted')
                        f1_std = 0.0
                        confusao = confusion_matrix(y_test, y_pred)
                        
                    elif training == "crossvalidation":
                        kf = KFold(n_splits=10, random_state=semente_aleatoria, shuffle=True)
                        
                        with warnings.catch_warnings(record=True) as w:
                            warnings.simplefilter("always")
                            scores = cross_val_score(model, X, y, scoring='f1_weighted', cv=kf)
                            y_pred = cross_val_predict(model, X, y, cv=kf)
                            
                            warning_exceptions = model_config.get('warning_exceptions', [])
                            has_warning = any(issubclass(warning.category, tuple(warning_exceptions)) for warning in w)
                        
                        confusao = confusion_matrix(y, y_pred)
                        f1 = scores.mean()
                        f1_std = scores.std()
                    
                    execution_time = time.time() - start_time
                    
                    if training == "holdout":
                        if has_warning:
                            print(f"\t\t\t⚠ WARNING: Problema detectado - F1: {f1:.4f} (Time: {execution_time:.4f}s)")
                        else:
                            print(f"\t\t\t✓ Sucesso - F1: {f1:.4f} (Time: {execution_time:.4f}s)")
                    else:
                        if has_warning:
                            print(f"\t\t\t⚠ WARNING: Problema detectado - F1: {f1:.4f} ({f1_std:.4f}) (Time: {execution_time:.4f}s)")
                        else:
                            print(f"\t\t\t✓ Sucesso - F1: {f1:.4f} ({f1_std:.4f}) (Time: {execution_time:.4f}s)")
                    
                    resultado = {
                        'dataset': metadata.key,
                        'model': model_key,
                        'model_name': model_config['model_name'],
                        'config_rank': config_rank,
                        'params': params,
                        'training_type': training,
                        'f1_score': f1,
                        'f1_std': f1_std,
                        'confusion_matrix': confusao,
                        'trained_model': model,
                        'scale_data': scale_data,
                        'execution_time': execution_time
                    }
                    resultados_todos_modelos.append(resultado)
                    
                except Exception as e:
                    print(f"\t\t\t✗ ERRO: {type(e).__name__}: {str(e)}")
                    continue
    
    print("-" * 80)

df_resultados_todos = pd.DataFrame(resultados_todos_modelos)

df_resultados_todos['shape'] = df_resultados_todos['dataset'].apply(
    lambda x: datafiles.loc[datafiles['key'] == x, 'shape'].values[0]
)

print("✅ Aplicação concluída!")
print(f"Total de combinações processadas: {len(df_resultados_todos)}")
print(f"Datasets processados: {df_resultados_todos['dataset'].nunique()}")
print(f"Modelos avaliados: {df_resultados_todos['model'].nunique()}")
print(f"Configurações por modelo: {df_resultados_todos['config_rank'].nunique()}")


Aplicando top 10 configurações de todos os modelos em todas as bases de dados...

📊 Dataset 1: hogfeat_128_16_4_9_pca
------------------------------------------------------------

🔧 Modelo: KNN (knn) Dataset 1: hogfeat_128_16_4_9_pca
		Escalamento: Não
		Training: holdout
			✓ Sucesso - F1: 0.7677 (Time: 0.0082s)
		Training: crossvalidation
			✓ Sucesso - F1: 0.7136 (0.0360) (Time: 0.0986s)
		Escalamento: Não
		Training: holdout
			✓ Sucesso - F1: 0.7677 (Time: 0.0085s)
		Training: crossvalidation
			✓ Sucesso - F1: 0.7136 (0.0360) (Time: 0.0934s)
		Escalamento: Não
		Training: holdout
			✓ Sucesso - F1: 0.7621 (Time: 0.0077s)
		Training: crossvalidation
			✓ Sucesso - F1: 0.7301 (0.0411) (Time: 0.1059s)
		Escalamento: Não
		Training: holdout
			✓ Sucesso - F1: 0.7621 (Time: 0.0082s)
		Training: crossvalidation
			✓ Sucesso - F1: 0.7301 (0.0411) (Time: 0.0939s)
		Escalamento: Não
		Training: holdout
			✓ Sucesso - F1: 0.7590 (Time: 0.0082s)
		Training: crossvalidation
			✓ Sucesso - F1

In [12]:
print("📈 Melhores F1 scores por dataset e modelo:")
for dataset in df_resultados_todos['dataset'].unique():
    print(f"\n  Dataset: {dataset}")
    dataset_results = df_resultados_todos[df_resultados_todos['dataset'] == dataset]
    for model in dataset_results['model'].unique():
        model_results = dataset_results[dataset_results['model'] == model]
        best_result = model_results.loc[model_results['f1_score'].idxmax()]
        print(f"    {model.upper()}: {best_result['f1_score']:.4f} (Config {best_result['config_rank']}, {best_result['training_type']})")


📈 Melhores F1 scores por dataset e modelo:

  Dataset: hogfeat_128_16_4_9_pca
    KNN: 0.7677 (Config 1, holdout)
    DTREE: 0.7567 (Config 1, holdout)
    NB: 0.7735 (Config 1, crossvalidation)
    MLP: 0.8068 (Config 1, holdout)

  Dataset: hogfeat_256_64_2_18
    KNN: 0.6276 (Config 1, holdout)
    DTREE: 0.6849 (Config 5, holdout)
    NB: 0.7508 (Config 2, holdout)
    MLP: 0.7999 (Config 3, crossvalidation)

  Dataset: hogfeat_256_64_2_9
    KNN: 0.5928 (Config 9, holdout)
    DTREE: 0.6996 (Config 10, crossvalidation)
    NB: 0.7757 (Config 2, holdout)
    MLP: 0.7914 (Config 10, crossvalidation)

  Dataset: hogfeat_128_16_4_9
    KNN: 0.5213 (Config 3, crossvalidation)
    DTREE: 0.6693 (Config 5, holdout)
    NB: 0.7382 (Config 2, holdout)
    MLP: 0.7895 (Config 1, crossvalidation)

  Dataset: hogfeat_256_32_2_9_pca
    KNN: 0.7452 (Config 5, holdout)
    DTREE: 0.7265 (Config 1, crossvalidation)
    NB: 0.7617 (Config 2, holdout)
    MLP: 0.7901 (Config 8, crossvalidation)

 

In [20]:

print("💾 Resultados salvos:")

# Preparar DataFrame para CSV expandindo confusion_matrix em colunas separadas
df_csv_resultados = df_resultados_todos.copy()

# Expandir confusion_matrix em 4 colunas separadas (sempre 2x2)
df_csv_resultados['cm_tp'] = df_csv_resultados['confusion_matrix'].apply(lambda x: x[0][0])
df_csv_resultados['cm_fp'] = df_csv_resultados['confusion_matrix'].apply(lambda x: x[0][1])
df_csv_resultados['cm_fn'] = df_csv_resultados['confusion_matrix'].apply(lambda x: x[1][0])
df_csv_resultados['cm_tn'] = df_csv_resultados['confusion_matrix'].apply(lambda x: x[1][1])

# Remover as colunas originais que não queremos no CSV
df_csv_resultados = df_csv_resultados.drop(['confusion_matrix', 'trained_model'], axis=1)

# Salvar CSV
df_csv_resultados.to_csv(f"resultados_top10_todos_modelos_{versao_execucao}.csv", index=False)
print(f"- resultados_top10_todos_modelos.csv: {len(df_csv_resultados)} registros (com colunas expandidas da matriz de confusão)")

# Salvar resultados completos com joblib
joblib.dump(df_resultados_todos, f'resultados_top10_todos_modelos_{versao_execucao}.joblib')
print(f"- resultados_top10_todos_modelos.joblib: dados completos com modelos treinados")

💾 Resultados salvos:
- resultados_top10_todos_modelos.csv: 792 registros (com colunas expandidas da matriz de confusão)
- resultados_top10_todos_modelos.joblib: dados completos com modelos treinados


## Salvando resultados em CSV e Joblib

### Amostra do DataFrame de Resultados

In [23]:
# Visualiza os resultados MLP
print(f"Total de combinações processadas: {len(df_resultados_todos)}")
print(f"Colunas disponíveis: {df_resultados_todos.columns.tolist()}")
print(f"Datasets processados: {df_resultados_todos['dataset'].nunique()}")
print(f"Configurações por dataset: {df_resultados_todos['config_rank'].nunique()}")

df_resultados_todos


Total de combinações processadas: 792
Colunas disponíveis: ['dataset', 'model', 'model_name', 'config_rank', 'params', 'training_type', 'f1_score', 'f1_std', 'confusion_matrix', 'trained_model', 'scale_data', 'execution_time', 'shape']
Datasets processados: 12
Configurações por dataset: 10


,dataset,model,model_name,config_rank,params,training_type,f1_score,f1_std,confusion_matrix,trained_model,scale_data,execution_time,shape
0,hogfeat_128_16_4_9_pca,knn,KNN,1,"{'metric': 'chebyshev', 'n_neighbors': 3, 'weights': 'distance'}",holdout,0.767689,0.000000,"[[50, 21], [16, 73]]","KNeighborsClassifier(metric='chebyshev', n_neighbors=3, weights='distance')",False,0.008203,"(800, 104)"
1,hogfeat_128_16_4_9_pca,knn,KNN,1,"{'metric': 'chebyshev', 'n_neighbors': 3, 'weights': 'distance'}",crossvalidation,0.713605,0.035977,"[[253, 147], [81, 319]]","KNeighborsClassifier(metric='chebyshev', n_neighbors=3, weights='distance')",False,0.098579,"(800, 104)"
2,hogfeat_128_16_4_9_pca,knn,KNN,2,"{'metric': 'chebyshev', 'n_neighbors': 3, 'weights': 'uniform'}",holdout,0.767689,0.000000,"[[50, 21], [16, 73]]","KNeighborsClassifier(metric='chebyshev', n_neighbors=3)",False,0.008464,"(800, 104)"
3,hogfeat_128_16_4_9_pca,knn,KNN,2,"{'metric': 'chebyshev', 'n_neighbors': 3, 'weights': 'uniform'}",crossvalidation,0.713605,0.035977,"[[253, 147], [81, 319]]","KNeighborsClassifier(metric='chebyshev', n_neighbors=3)",False,0.093410,"(800, 104)"
4,hogfeat_128_16_4_9_pca,knn,KNN,3,"{'metric': 'chebyshev', 'n_neighbors': 5, 'weights': 'distance'}",holdout,0.762123,0.000000,"[[51, 20], [18, 71]]","KNeighborsClassifier(metric='chebyshev', weights='distance')",False,0.007729,"(800, 104)"
...,...,...,...,...,...,...,...,...,...,...,...,...,...
787,lbpfeat_256_24_192,mlp,MLP,8,"{'activation': 'relu', 'hidden_layer_sizes': (200, 100, 50), 'learning_rate_init': 0.005, 'max_iter': 500, 'solver': 'adam'}",crossvalidation,0.730035,0.067139,"[[279, 121], [94, 306]]","MLPClassifier(hidden_layer_sizes=(200, 100, 50), learning_rate_init=0.005,\n max_iter=500, random_state=42)",False,7.179003,"(800, 195)"
788,lbpfeat_256_24_192,mlp,MLP,9,"{'activation': 'relu', 'hidden_layer_sizes': 150, 'learning_rate_init': 0.01, 'max_iter': 500, 'solver': 'adam'}",holdout,0.781638,0.000000,"[[55, 16], [19, 70]]","MLPClassifier(hidden_layer_sizes=150, learning_rate_init=0.01, max_iter=500,\n random_state=42)",False,0.262154,"(800, 195)"
789,lbpfeat_256_24_192,mlp,MLP,9,"{'activation': 'relu', 'hidden_layer_sizes': 150, 'learning_rate_init': 0.01, 'max_iter': 500, 'solver': 'adam'}",crossvalidation,0.743986,0.053737,"[[285, 115], [90, 310]]","MLPClassifier(hidden_layer_sizes=150, learning_rate_init=0.01, max_iter=500,\n random_state=42)",False,4.680442,"(800, 195)"
790,lbpfeat_256_24_192,mlp,MLP,10,"{'activation': 'relu', 'hidden_layer_sizes': (200, 100), 'learning_rate_init': 0.001, 'max_iter': 1500, 'solver': 'adam'}",holdout,0.798849,0.000000,"[[52, 19], [13, 76]]","MLPClassifier(hidden_layer_sizes=(200, 100), max_iter=1500, random_state=42)",False,0.634410,"(800, 195)"
